<a href="https://colab.research.google.com/github/Rakib911Hossan/ml-penta/blob/dev/machine_translation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import requests
import json
import os
import regex as re

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [2]:
def download_file(url, path):
    if not os.path.exists(path):
        r = requests.get(url); r.raise_for_status()
        with open(path, "wb") as f: f.write(r.content)

download_file("https://openaipublic.blob.core.windows.net/gpt-2/models/124M/vocab.bpe",  "vocab.bpe")
download_file("https://openaipublic.blob.core.windows.net/gpt-2/models/124M/encoder.json","encoder.json")

with open("encoder.json") as f: encoder = json.load(f)
decoder = {v: k for k, v in encoder.items()}

with open("vocab.bpe", encoding="utf-8") as f:
    bpe_data = f.read()
bpe_merges = [tuple(m.split()) for m in bpe_data.split("\n")[1:-1]]
bpe_ranks  = {m: i for i, m in enumerate(bpe_merges)}

def bytes_to_unicode():
    bs = list(range(ord("!"), ord("~")+1)) + \
         list(range(ord("¡"), ord("¬")+1)) + \
         list(range(ord("®"), ord("ÿ")+1))
    cs = bs[:]
    n = 0
    for b in range(2**8):
        if b not in bs:
            bs.append(b); cs.append(2**8 + n); n += 1
    return {b: chr(c) for b, c in zip(bs, cs)}

byte_encoder = bytes_to_unicode()
byte_decoder = {v: k for k, v in byte_encoder.items()}
pat = re.compile(r"'s|'t|'re|'ve|'m|'ll|'d| ?\w+| ?\d+| ?[^\s\w\d]+|\s+(?!\S)|\s+")

def bpe(token):
    word = tuple(token)
    pairs = {(word[i], word[i+1]) for i in range(len(word)-1)}
    if not pairs: return token
    while True:
        bigram = min(pairs, key=lambda p: bpe_ranks.get(p, float("inf")))
        if bigram not in bpe_ranks: break
        first, second = bigram
        new_word, i = [], 0
        while i < len(word):
            try:
                j = word.index(first, i)
                new_word.extend(word[i:j]); i = j
            except ValueError:
                new_word.extend(word[i:]); break
            if i < len(word)-1 and word[i+1] == second:
                new_word.append(first+second); i += 2
            else:
                new_word.append(word[i]); i += 1
        word = tuple(new_word)
        if len(word) == 1: break
        pairs = {(word[i], word[i+1]) for i in range(len(word)-1)}
    return " ".join(word)

bpe_cache = {}
def encode(text):
    ids = []
    for token in re.findall(pat, text):
        token = "".join(byte_encoder[b] for b in token.encode("utf-8"))
        if token not in bpe_cache: bpe_cache[token] = bpe(token)
        ids.extend(encoder[t] for t in bpe_cache[token].split())
    return ids

def decode(ids):
    text = "".join(decoder[i] for i in ids)
    return bytearray([byte_decoder[c] for c in text]).decode("utf-8", errors="replace")

NEWLINE_ID = encode("\n")[0]
EOS_ID = 50256

In [3]:
class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_head = cfg["n_head"]
        self.n_embd = cfg["n_embd"]
        self.c_attn = nn.Linear(cfg["n_embd"], 3*cfg["n_embd"])
        self.c_proj = nn.Linear(cfg["n_embd"],   cfg["n_embd"])
        self.register_buffer("bias",
            torch.tril(torch.ones(cfg["n_ctx"], cfg["n_ctx"]))
                  .view(1,1,cfg["n_ctx"],cfg["n_ctx"]))

    # FIX 1a: now accepts attention_mask
    def forward(self, x, attention_mask=None):
        B, T, C = x.shape
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        nh, hs  = self.n_head, C // self.n_head
        def split(t): return t.view(B,T,nh,hs).transpose(1,2)
        q, k, v = split(q), split(k), split(v)
        att = (q @ k.transpose(-2,-1)) / (hs**0.5)
        att = att.masked_fill(self.bias[:,:,:T,:T]==0, float("-inf"))
        if attention_mask is not None:
            pad_mask = (attention_mask == 0).unsqueeze(1).unsqueeze(2)
            att = att.masked_fill(pad_mask, float("-inf"))
        att = F.softmax(att, dim=-1)
        return self.c_proj((att @ v).transpose(1,2).contiguous().view(B,T,C))

class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.c_fc   = nn.Linear(cfg["n_embd"], 4*cfg["n_embd"])
        self.c_proj = nn.Linear(4*cfg["n_embd"], cfg["n_embd"])
    def forward(self, x):
        return self.c_proj(F.gelu(self.c_fc(x), approximate="tanh"))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln_1 = nn.LayerNorm(cfg["n_embd"])
        self.attn  = CausalSelfAttention(cfg)
        self.ln_2 = nn.LayerNorm(cfg["n_embd"])
        self.mlp   = MLP(cfg)

    # FIX 1b: passes attention_mask to attention
    def forward(self, x, attention_mask=None):
        x = x + self.attn(self.ln_1(x), attention_mask=attention_mask)
        x = x + self.mlp(self.ln_2(x))
        return x

class GPT2(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.wte  = nn.Embedding(cfg["vocab_size"], cfg["n_embd"])
        self.wpe  = nn.Embedding(cfg["n_ctx"],      cfg["n_embd"])
        self.h    = nn.ModuleList([Block(cfg) for _ in range(cfg["n_layer"])])
        self.ln_f = nn.LayerNorm(cfg["n_embd"])
        self.lm_head = nn.Linear(cfg["n_embd"], cfg["vocab_size"], bias=False)
        self.lm_head.weight = self.wte.weight

    # FIX 1c: forward passes attention_mask through all blocks
    def forward(self, idx, attention_mask=None):
        B, T = idx.shape
        pos  = torch.arange(T, device=idx.device)
        x    = self.wte(idx) + self.wpe(pos)
        for block in self.h:
            x = block(x, attention_mask=attention_mask)
        return self.lm_head(self.ln_f(x))

In [4]:
MODEL_DIR = "gpt2_weights"
os.makedirs(MODEL_DIR, exist_ok=True)
BASE = "https://openaipublic.blob.core.windows.net/gpt-2/models/124M/"
for fname in ["checkpoint","hparams.json","model.ckpt.data-00000-of-00001",
              "model.ckpt.index","model.ckpt.meta"]:
    download_file(BASE+fname, os.path.join(MODEL_DIR, fname))

import tensorflow as tf

def load_weights(model, model_dir):
    ckpt = os.path.join(model_dir, "model.ckpt")
    def t(name): return torch.tensor(tf.train.load_variable(ckpt, name))

    model.wte.weight.data  = t("model/wte")
    model.wpe.weight.data  = t("model/wpe")
    model.ln_f.weight.data = t("model/ln_f/g")
    model.ln_f.bias.data   = t("model/ln_f/b")

    for i, block in enumerate(model.h):
        p = f"model/h{i}"
        block.ln_1.weight.data        = t(f"{p}/ln_1/g")
        block.ln_1.bias.data          = t(f"{p}/ln_1/b")
        block.ln_2.weight.data        = t(f"{p}/ln_2/g")
        block.ln_2.bias.data          = t(f"{p}/ln_2/b")
        block.attn.c_attn.weight.data = t(f"{p}/attn/c_attn/w")[0].T
        block.attn.c_attn.bias.data   = t(f"{p}/attn/c_attn/b")
        block.attn.c_proj.weight.data = t(f"{p}/attn/c_proj/w")[0].T
        block.attn.c_proj.bias.data   = t(f"{p}/attn/c_proj/b")
        block.mlp.c_fc.weight.data    = t(f"{p}/mlp/c_fc/w")[0].T
        block.mlp.c_fc.bias.data      = t(f"{p}/mlp/c_fc/b")
        block.mlp.c_proj.weight.data  = t(f"{p}/mlp/c_proj/w")[0].T
        block.mlp.c_proj.bias.data    = t(f"{p}/mlp/c_proj/b")

    print("Weights loaded.")

cfg = {
    "vocab_size": 50257,
    "n_ctx":      1024,
    "n_embd":     768,
    "n_head":     12,
    "n_layer":    12,
}

model = GPT2(cfg).to(device)
load_weights(model, MODEL_DIR)
model.eval()

Weights loaded.


GPT2(
  (wte): Embedding(50257, 768)
  (wpe): Embedding(1024, 768)
  (h): ModuleList(
    (0-11): 12 x Block(
      (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (attn): CausalSelfAttention(
        (c_attn): Linear(in_features=768, out_features=2304, bias=True)
        (c_proj): Linear(in_features=768, out_features=768, bias=True)
      )
      (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): MLP(
        (c_fc): Linear(in_features=768, out_features=3072, bias=True)
        (c_proj): Linear(in_features=3072, out_features=768, bias=True)
      )
    )
  )
  (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [7]:
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader

dataset = load_dataset("opus100", "en-fr", streaming=True)

def make_prompt(fr, en):
    return f"Translate French to English.\nFrench: {fr}\nEnglish: {en}"

class TranslationDataset(Dataset):
    # FIX 3: max_samples raised 20000 -> 50000
    def __init__(self, split, max_samples=50000, max_len=128):
        self.data = list(dataset[split].take(max_samples))
        self.data = [
            {"fr": x["translation"]["fr"], "en": x["translation"]["en"]}
            for x in self.data
        ]
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        pair = self.data[idx]
        prompt_only = f"Translate French to English.\nFrench: {pair['fr']}\nEnglish: "
        full        = make_prompt(pair['fr'], pair['en'])

        input_ids  = encode(full)[:self.max_len -1] + [EOS_ID]
        prompt_ids_only = encode(prompt_only)
        prompt_len = min(len(prompt_ids_only), len(input_ids) - 1)
        labels     = [-100] * prompt_len + input_ids[prompt_len:]

        return input_ids, labels

def collate_fn(batch):
    input_ids_list, labels_list = zip(*batch)
    max_len = max(len(x) for x in input_ids_list)

    input_ids_padded = []
    labels_padded    = []
    attention_masks  = []

    for ids, labs in zip(input_ids_list, labels_list):
        pad_len = max_len - len(ids)
        input_ids_padded.append(ids + [0] * pad_len)
        labels_padded.append(labs + [-100] * pad_len)
        attention_masks.append([1] * len(ids) + [0] * pad_len)

    return (
        torch.tensor(input_ids_padded, dtype=torch.long),
        torch.tensor(labels_padded,    dtype=torch.long),
        torch.tensor(attention_masks,  dtype=torch.long),
    )

train_loader = DataLoader(
    TranslationDataset("train"),
    batch_size=64,
    shuffle=True,
    collate_fn=collate_fn,
)

In [8]:
from torch.optim.lr_scheduler import CosineAnnealingLR

EPOCHS    = 10  # FIX 3: was 5
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler    = torch.amp.GradScaler('cuda')  # FIX 5: updated deprecated API
model = model.to(device)
model.train()

for epoch in range(EPOCHS):
    total_loss = 0

    # FIX 1d: unpack attention_mask instead of discarding with _
    for step, (input_ids, labels, attention_mask) in enumerate(train_loader):
        input_ids      = input_ids.to(device)
        labels         = labels.to(device)
        attention_mask = attention_mask.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):  # FIX 5: updated deprecated API
            # FIX 1d: pass attention_mask to model
            logits       = model(input_ids, attention_mask=attention_mask)
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=-100,
            )

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        if step % 100 == 0:
            print(f"Epoch {epoch+1} | Step {step} | Loss {loss.item():.4f}")

    avg = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} done | Avg Loss {avg:.4f} | LR {scheduler.get_last_lr()[0]:.2e}")
    scheduler.step()

torch.save(model.state_dict(), "gpt2_finetuned_fr_en.pt")
print("Model saved.")

Epoch 1 | Step 0 | Loss 3.8455
Epoch 1 | Step 100 | Loss 2.9387
Epoch 1 | Step 200 | Loss 3.0502
Epoch 1 | Step 300 | Loss 2.9177
Epoch 1 | Step 400 | Loss 2.6299
Epoch 1 | Step 500 | Loss 2.7128
Epoch 1 | Step 600 | Loss 2.7356
Epoch 1 | Step 700 | Loss 2.5270
Epoch 1 done | Avg Loss 2.7599 | LR 3.00e-05
Epoch 2 | Step 0 | Loss 2.3898
Epoch 2 | Step 100 | Loss 2.1601
Epoch 2 | Step 200 | Loss 2.3280
Epoch 2 | Step 300 | Loss 2.2924
Epoch 2 | Step 400 | Loss 2.0785
Epoch 2 | Step 500 | Loss 2.3310
Epoch 2 | Step 600 | Loss 2.3540
Epoch 2 | Step 700 | Loss 1.9584
Epoch 2 done | Avg Loss 2.2034 | LR 2.93e-05
Epoch 3 | Step 0 | Loss 1.8862
Epoch 3 | Step 100 | Loss 1.7299
Epoch 3 | Step 200 | Loss 1.6370
Epoch 3 | Step 300 | Loss 1.9525
Epoch 3 | Step 400 | Loss 2.0178
Epoch 3 | Step 500 | Loss 1.7833
Epoch 3 | Step 600 | Loss 1.7455
Epoch 3 | Step 700 | Loss 1.9182
Epoch 3 done | Avg Loss 1.8978 | LR 2.71e-05
Epoch 4 | Step 0 | Loss 1.6030
Epoch 4 | Step 100 | Loss 1.6207
Epoch 4 | Step 

In [9]:
def generate_beam(prompt, max_new_tokens=60, beam_width=5, length_penalty=0.7, repetition_penalty=1.3):
    model.to(device)
    prompt_ids = encode(prompt)
    beams      = [(0.0, prompt_ids)]
    completed  = []

    with torch.no_grad():
        for _ in range(max_new_tokens):
            all_candidates = []

            for score, ids in beams:
                if len(ids) > len(prompt_ids) and ids[-1] == EOS_ID:
                    completed.append((score, ids))
                    continue

                input_ids = torch.tensor([ids[-1024:]], dtype=torch.long, device=device)
                logits    = model(input_ids)[:, -1, :]

                for token_id in set(ids):
                    if logits[0, token_id] > 0:
                        logits[0, token_id] /= repetition_penalty
                    else:
                        logits[0, token_id] *= repetition_penalty

                log_probs                 = F.log_softmax(logits, dim=-1)
                top_log_probs, top_tokens = torch.topk(log_probs, beam_width)

                for i in range(beam_width):
                    token     = top_tokens[0, i].item()
                    new_score = score + top_log_probs[0, i].item()
                    all_candidates.append((new_score, ids + [token]))

            if not all_candidates:
                break

            all_candidates.sort(key=lambda x: x[0] / (len(x[1]) ** length_penalty), reverse=True)
            beams = all_candidates[:beam_width]


            if all(ids[-1] == EOS_ID for _, ids in beams):
                completed.extend(beams)
                break

    final = completed if completed else beams
    final.sort(key=lambda x: x[0] / (len(x[1]) ** length_penalty), reverse=True)
    return final[0][1]


def translate_beam(text):
    prompt = f"Translate French to English.\nFrench: {text}\nEnglish: "
    prompt_token_len = len(encode(prompt))
    out_ids = generate_beam(prompt, max_new_tokens=60, beam_width=5)
    translation_ids = out_ids[prompt_token_len:]

    # strip EOS if present
    if EOS_ID in translation_ids:
        translation_ids = translation_ids[:translation_ids.index(EOS_ID)]

    return decode(translation_ids).strip()


tests = [
    "Bonjour, comment ça va ?",
    "Je mange une pomme.",
    "La vie est belle.",
    "Le parlement européen se réunit aujourd'hui.",
]

model.eval()
for sentence in tests:
    print(f"FR: {sentence}")
    print(f"EN: {translate_beam(sentence)}\n")

FR: Bonjour, comment ça va ?
EN: What's going on?

FR: Je mange une pomme.
EN: _________________

FR: La vie est belle.
EN: The life is beautiful

FR: Le parlement européen se réunit aujourd'hui.
EN: The European Parliament has postponed today's vote for some time

